# D-Fire YOLOv8n — Colab 학습

다음 E1 실험을 같은 데이터·학습 설정으로 Colab GPU에서 새로 시작하는 노트북입니다.
`런타임 → 런타임 유형 변경 → GPU`를 선택한 뒤 위에서부터 실행하세요.
Google 계정 연결과 실제 실행은 사용자가 Colab에서 수행합니다.

먼저 PC에 준비된 `dfire-fold1-v2.zip`을 Google Drive의
`내 드라이브/drone-fire-detection/`에 업로드하세요.
이미지·라벨은 Colab 로컬 디스크에서 읽고, 학습·평가 결과와 checkpoint는 Drive에 저장합니다.
원본 데이터·모델 가중치·환경을 GitHub에 업로드하지 않습니다.

Windows E0와 Colab E1은 GPU·OS·Python이 달라질 수 있습니다. 엄격한 epoch 비교에는
Colab에서도 E0와 E1을 같은 실행 환경으로 학습하는 것이 좋습니다. 아래 ID를 E0로 바꾸면 됩니다.

Colab의 GPU 종류와 사용 가능 시간은 가용성·요금제에 따라 달라집니다.
무료 GPU가 RTX 5070보다 빠르다고 보장할 수 없습니다.
이 노트북에서는 Test 평가를 실행하지 않습니다.

출처: [Colab FAQ](https://research.google.com/colaboratory/faq.html),
[프로젝트 사용 방법](https://github.com/Zero-woo/drone-fire-detection/blob/main/docs/USAGE.md).

## 1. 실행 설정

기본 실험은 E1(100 epoch)입니다. 아래 실험 ID만 바꾸어 개별 실험을 실행할 수 있습니다.
`RESUME_RUN_NAME`은 Colab에서 중단된 동일 실험을 재개할 때만 지정합니다.
PC의 Windows checkpoint 경로를 이 칸에 넣어 이동 재개하는 기능은 포함하지 않습니다.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/Zero-woo/drone-fire-detection.git"
REPO_REF = "162c21d15ff06e41a01c9896cd7c097acffef5f1"  # Pinned Colab implementation.
REPO = Path("/content/drone-fire-detection")
DRIVE_FOLDER = Path("/content/drive/MyDrive/drone-fire-detection")
ARCHIVE_NAME = "dfire-fold1-v2.zip"
ARCHIVE_SHA256 = "cb6c4abbaca83496d4dc578ec804b3e5b5d839ce7c89f1fe58ffc66e4f6a0476"
EXPECTED_DATASET_FINGERPRINT = "c3d63dfa73314a36b5522f4651f5fd7d784bd8a054e78916f821160729862e2b"
EXPERIMENT_ID = "E1"
RESUME_RUN_NAME = ""  # Example: E1_20261031T150000Z_1234abcd

if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("This pinned environment requires Python 3.10-3.13; choose a compatible Colab runtime")

def run(arguments, cwd=None, env=None):
    # Stream ordinary output and throttle frequent progress lines to keep the notebook usable.
    process = subprocess.Popen([str(x) for x in arguments], cwd=cwd, env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", bufsize=1)
    last_progress = 0.0
    try:
        for line in process.stdout:
            progress = "it/s" in line or "s/it" in line or "━" in line
            now = time.monotonic()
            if not progress or now - last_progress >= 5:
                print(line, end="", flush=True)
                if progress:
                    last_progress = now
        result = process.wait()
    except KeyboardInterrupt:
        import signal
        process.send_signal(signal.SIGINT)
        process.wait(timeout=60)
        raise
    if result:
        raise subprocess.CalledProcessError(result, arguments)

## 2. 소스 가져오기와 전용 가상환경 설치

모든 프로젝트 패키지는 저장소의 `.venv`에 설치합니다.
Colab 기본 Python에는 프로젝트 의존성을 설치하지 않습니다.
CUDA 12.8 PyTorch wheel과 현재 프로젝트의 고정 버전을 사용합니다.
설치·GPU 검증이 실패하면 이후 셀을 실행하지 말고 오류를 확인하세요.

In [ ]:
if not (REPO / ".git").is_dir():
    run(["git", "clone", REPO_URL, REPO])
origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
if origin != REPO_URL:
    raise RuntimeError("Existing checkout uses another remote")
dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
if dirty:
    raise RuntimeError("Checkout has source edits; review them before switching commits")
run(["git", "-C", REPO, "checkout", "--detach", REPO_REF])
print("Source commit:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
PYTHON = REPO / ".venv/bin/python"
if not PYTHON.is_file():
    run([sys.executable, "-m", "venv", "--without-pip", REPO / ".venv"])
    # pip --python manages the target venv even when that venv has no pip yet.
    run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "pip"])
run([PYTHON, "-m", "pip", "install", "torch==2.10.0", "torchvision==0.25.0",
     "--index-url", "https://download.pytorch.org/whl/cu128"], cwd=REPO)
run([PYTHON, "-m", "pip", "install", "-r", "requirements.txt", "-c", "requirements-lock.txt"], cwd=REPO)
run([PYTHON, "-m", "pip", "check"], cwd=REPO)
run([PYTHON, "firelab.py", "doctor", "--require-gpu"], cwd=REPO)
run([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=REPO)

## 3. Google Drive 연결 및 영구 결과 폴더

각 epoch의 checkpoint와 실험 기록이 Drive에 저장됩니다.
런타임이 종료되면 `/content`는 사라질 수 있으므로 입력 ZIP도 Drive에 보관하세요.
실행 중 Drive 저장이 실패하면 오류를 확인하고 마지막 정상 checkpoint를 사용해야 합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)
RUNS = DRIVE_FOLDER / "runs"
RUNS.mkdir(exist_ok=True)
TRAIN_ENV = os.environ.copy()
TRAIN_ENV["FIRELAB_RUNS_DIR"] = str(RUNS)
print("Persistent runs:", RUNS)

## 4. 검증된 데이터 이동

Drive에서 단일 ZIP을 `/content`로 복사한 뒤 풉니다. Drive의 수만 개 이미지를 직접 읽지 않습니다.
ZIP SHA256, 이미지·라벨 SHA256과 데이터 fingerprint가 맞지 않으면 중단합니다.
Windows 절대 경로가 들어 있던 split 목록/YAML만 Colab 경로로 다시 만듭니다.
픽셀·라벨·split 구성과 데이터 fingerprint는 PC의 E0와 동일하게 유지합니다.
같은 런타임에서 이미 정상 import했다면 재사용합니다. `validate-data`를 다시 실행할 필요가 없습니다.

In [ ]:
source_archive = DRIVE_FOLDER / ARCHIVE_NAME
if not source_archive.is_file():
    raise FileNotFoundError(f"Upload the prepared ZIP first: {source_archive}")
local_archive = REPO / "work" / ARCHIVE_NAME
local_archive.parent.mkdir(parents=True, exist_ok=True)
if not local_archive.exists():
    shutil.copyfile(source_archive, local_archive)
run([PYTHON, "firelab.py", "import-colab", "--archive", local_archive,
     "--sha256", ARCHIVE_SHA256], cwd=REPO)
dataset_config = json.loads((REPO / "configs/dataset.json").read_text())
manifest = json.loads((REPO / dataset_config["prepared_root"] / "manifest.json").read_text())
if manifest["fingerprint"] != EXPECTED_DATASET_FINGERPRINT:
    raise RuntimeError("Dataset identity differs from the PC baseline")
print("Verified dataset:", manifest["fingerprint"])
print(json.dumps(manifest["summary"], indent=2))
run([PYTHON, "firelab.py", "fetch-models"], cwd=REPO)
run([PYTHON, "firelab.py", "plan", "--id", EXPERIMENT_ID], cwd=REPO)

## 5. 단일 실험 학습 또는 동일 실험 재개

새 실험은 COCO 가중치에서 시작합니다. E0의 best.pt를 이어서 E1을 학습하지 않습니다.
기본 설정은 FP32·workers 0 등 PC와 같은 조건입니다. GPU가 다르므로 속도는 직접 측정해야 합니다.

런타임이 끊긴 경우 1~4단계를 다시 실행한 뒤, 설정 셀의 `RESUME_RUN_NAME`에
Drive `runs/` 아래의 중단된 폴더 이름을 입력하고 이 셀을 실행하세요.
checkpoint가 정상 저장되기 전의 epoch 진행분은 복구되지 않습니다.

In [ ]:
if RESUME_RUN_NAME:
    if Path(RESUME_RUN_NAME).name != RESUME_RUN_NAME:
        raise ValueError("Supply a run folder name, not an absolute path")
    TRAINING_RUN = RUNS / RESUME_RUN_NAME
    run([PYTHON, "firelab.py", "resume", "--run", TRAINING_RUN], cwd=REPO, env=TRAIN_ENV)
else:
    before = set(RUNS.glob(EXPERIMENT_ID + "_*"))
    try:
        run([PYTHON, "firelab.py", "train", "--id", EXPERIMENT_ID], cwd=REPO, env=TRAIN_ENV)
    finally:
        created = set(RUNS.glob(EXPERIMENT_ID + "_*")) - before
        if len(created) == 1:
            TRAINING_RUN = created.pop()
            print("Training run:", TRAINING_RUN)
training_record = json.loads((TRAINING_RUN / "training.json").read_text())
print("Status:", training_record["status"])
print("Training seconds:", training_record.get("training_wall_seconds"))

## 6. Validation 평가

모델 선택에는 Validation만 사용합니다. Test는 최종 모델과 threshold를 확정한 뒤 평가합니다.
공통 평가에는 PC와 같은 정사각형 padding, AP confidence 0.001과 운영 confidence 0.25를 사용합니다.
결과의 CUDA/GPU·라이브러리 버전과 학습 시간을 함께 확인하세요.

In [ ]:
if training_record["status"] != "completed":
    raise RuntimeError("Complete/resume training before evaluation")
before = set(RUNS.glob("val-" + training_record["experiment_id"] + "_*"))
run([PYTHON, "firelab.py", "evaluate", "--run", TRAINING_RUN], cwd=REPO, env=TRAIN_ENV)
created = set(RUNS.glob("val-" + training_record["experiment_id"] + "_*")) - before
if len(created) != 1:
    raise RuntimeError("Could not identify the new validation run")
EVALUATION_RUN = created.pop()
evaluation = json.loads((EVALUATION_RUN / "evaluation.json").read_text())
print("Evaluation run:", EVALUATION_RUN)
print(json.dumps({"metrics": evaluation.get("metrics"), "operating": evaluation.get("operating")}, indent=2))
print("Best weights:", training_record["best_path"])

## 결과 보관과 한계

Drive `drone-fire-detection/runs/`에 `training.json`, `train/results.csv`,
`train/weights/best.pt`, `last.pt`, 평가 JSON과 사례 이미지가 남습니다.
필요한 run 폴더를 PC로 다운로드해 결과 분석에 사용할 수 있습니다.
PC에서 재개·평가하려면 기록된 Colab 절대 경로를 옮기는 절차가 추가로 필요합니다.

이 노트북은 로컬에서 코드 구문·데이터 이전·checkpoint 경로 설정을 검증한 실행 준비물입니다.
실제 Colab GPU 할당, 설치 성공 여부와 학습 속도는 해당 Colab 세션에서 확인해야 합니다.
GPU가 달라지면 미세한 수치 차이가 날 수 있으며 속도는 동일 하드웨어 비교와 구분합니다.
유료 요금제 가입이나 compute unit 구매를 자동으로 수행하지 않습니다.